# Chapter 2: Preparing Insurance Data with Python

Code of Chapter 2 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

## 2.1.3 The example data

In [ ]:
import pandas as pd
from gidata import exposure_table, claim_position, check_claims

policies = pd.DataFrame({
    "policy_id": ["P001", "P002", "P003", "P004", "P005", "P006"],
    "start":  ["2025-01-01", "2025-03-15", "2025-07-01", "2025-10-01", "2025-05-20", "2025-12-01"],
    "end":    ["2025-12-31", "2026-03-14", "2026-06-30", "2026-09-30", "2026-05-19", "2026-11-30"],
    "cancel": [None, None, None, None, "2025-09-01", None],
    "region": ["North", "City", "South", "City", "North", "South"],
    "premium": [730.0, 584.0, 900.0, 1095.0, 650.0, 365.0],
})

## 2.2.4 Worked example: the 2025 premium position

In [ ]:
t = exposure_table(policies, 2025)
t[["written_premium", "earned_exposure", "earned_premium", "unearned_premium"]].sum()
# 3859.21, 2.9260, 2143.10, 1716.10

## 2.2.5 Calendar-year split

In [ ]:
exposure_table(policies, 2026)[["earned_exposure", "earned_premium"]].sum()
# 2.3589, 1716.10

## 2.3.1 Claims and transactions in the example

In [ ]:
claims = pd.DataFrame({
    "claim_id": ["C01", "C02", "C03", "C04", "C05"],
    "policy_id": ["P001", "P002", "P003", "P004", "P005"],
    "accident_date": ["2025-02-10", "2025-06-03", "2025-08-20", "2025-11-15", "2025-09-10"],
    "report_date":   ["2025-02-12", "2025-06-20", "2025-08-22", "2025-11-05", "2025-09-12"],
})

In [ ]:
tx = pd.DataFrame({
    "claim_id": ["C01", "C01", "C01", "C02", "C02", "C02", "C02", "C03", "C03", "C04"],
    "date": ["2025-02-12", "2025-03-01", "2025-04-15", "2025-06-20", "2025-09-30",
             "2025-09-30", "2026-01-20", "2025-08-22", "2025-10-05", "2025-11-20"],
    "type": ["reserve", "payment", "reserve", "reserve", "payment",
             "reserve", "payment", "reserve", "reserve", "reserve"],
    "amount": [1500, 1200, 0, 8000, 3000, 5500, 2500, 2000, 3500, 4000],
})

## 2.3.2 The position at a valuation date

In [ ]:
claim_position(tx, "2025-12-31").sum()
# paid 4200.0, case_reserve 13000.0, incurred 17200.0

## 2.4.2 Running the checks on the example

In [ ]:
check_claims(claims, policies)
#   claim_id                     issue
# 0      C04  reported before accident
# 1      C05    accident outside cover

## 2.4.3 Summaries and groupings

In [ ]:
t.groupby("region")[["earned_exposure", "earned_premium"]].sum()

## 2.5.2 The model

In [ ]:
from gidata import synthetic_portfolio
pol, cl = synthetic_portfolio(n_policies=5_000, seed=2025)
len(pol), len(cl)                          # 5000 policies, 485 claims

## 2.5.4 Payments and case reserves

In [ ]:
from gidata import synthetic_transactions
tx_s = synthetic_transactions(cl, seed=2026)
len(tx_s)                                  # 2827 transactions

In [ ]:
acc25 = cl.loc[cl["accident_date"] <= "2025-12-31", "claim_id"]
t25 = tx_s[tx_s["claim_id"].isin(acc25)]
claim_position(t25, "2025-12-31").sum()    # paid 330238, case 416556, incurred 746794

## Exercise 1: A policy that contains 29 February

In [ ]:
leap = pd.DataFrame({"policy_id": ["L1"], "start": ["2027-09-01"],
                     "end": ["2028-08-31"], "premium": [732.0]})
exposure_table(leap, 2027)[["earned_exposure", "earned_premium", "unearned_premium"]]
# 0.3342, 244.0, 488.0

## Exercise 2: The claim position at an earlier date

In [ ]:
claim_position(tx, "2025-09-30").sum()
# paid 4200.0, case_reserve 7500.0, incurred 11700.0

## Exercise 3: Frequency by vehicle group in the synthetic portfolio

In [ ]:
e = exposure_table(pol, 2025)
expo = e.groupby("vehicle")["earned_exposure"].sum()
acc25 = cl[cl["accident_date"] <= "2025-12-31"].merge(pol[["policy_id", "vehicle"]])
freq = acc25.groupby("vehicle")["claim_id"].count() / expo
freq / freq["A"]                           # 1.000, 1.028, 1.513

## 2.8 Python Walkthrough

In [ ]:
import pandas as pd
from gidata import exposure_table, claim_position, check_claims, synthetic_portfolio

# Setup: the data and objects built in the sections of this chapter
policies = pd.DataFrame({
    "policy_id": ["P001", "P002", "P003", "P004", "P005", "P006"],
    "start":  ["2025-01-01", "2025-03-15", "2025-07-01", "2025-10-01", "2025-05-20", "2025-12-01"],
    "end":    ["2025-12-31", "2026-03-14", "2026-06-30", "2026-09-30", "2026-05-19", "2026-11-30"],
    "cancel": [None, None, None, None, "2025-09-01", None],
    "region": ["North", "City", "South", "City", "North", "South"],
    "premium": [730.0, 584.0, 900.0, 1095.0, 650.0, 365.0],
})
claims = pd.DataFrame({
    "claim_id": ["C01", "C02", "C03", "C04", "C05"],
    "policy_id": ["P001", "P002", "P003", "P004", "P005"],
    "accident_date": ["2025-02-10", "2025-06-03", "2025-08-20", "2025-11-15", "2025-09-10"],
    "report_date":   ["2025-02-12", "2025-06-20", "2025-08-22", "2025-11-05", "2025-09-12"],
})
tx = pd.DataFrame({
    "claim_id": ["C01", "C01", "C01", "C02", "C02", "C02", "C02", "C03", "C03", "C04"],
    "date": ["2025-02-12", "2025-03-01", "2025-04-15", "2025-06-20", "2025-09-30",
             "2025-09-30", "2026-01-20", "2025-08-22", "2025-10-05", "2025-11-20"],
    "type": ["reserve", "payment", "reserve", "reserve", "payment",
             "reserve", "payment", "reserve", "reserve", "reserve"],
    "amount": [1500, 1200, 0, 8000, 3000, 5500, 2500, 2000, 3500, 4000],
})

# Section 2.2: exposure and premium by calendar year
t25 = exposure_table(policies, 2025)
t25[["written_premium", "earned_exposure", "earned_premium", "unearned_premium"]].sum()
# 3859.21, 2.9260, 2143.10, 1716.10
exposure_table(policies, 2026)["earned_premium"].sum()     # 1716.10 = 2025 UPR

# Section 2.3: claim position at a valuation date
claim_position(tx, "2025-12-31")["incurred"].sum()           # 17200.0
claim_position(tx, "2026-01-31")["incurred"].sum()           # 19700.0

# Section 2.4: data checks and groupings
check_claims(claims, policies)                               # C04, C05 flagged
t25.groupby("region")["earned_exposure"].sum()               # City 1.0521, North 1.2849, South 0.5890

# Section 2.5: synthetic portfolio
pol, cl = synthetic_portfolio(seed=2025)
exposure_table(pol, 2025)["earned_exposure"].sum()           # 2508.25
cl["ultimate_amount"].mean()                                 # 3873.94